# 05 · Tail-robust effect sizes: bounded-loss scoring and common support

**Failure being addressed** (notebook 04, validation B4/B5). With the flexible context (`spline+proj`), the zero-information test is valid on fresh worlds, but the *mean* held-out log-score contrast is not. In mixed-cue worlds a few held-out rows at the temperature extremes score −117 to −130 nats, because the joint model extrapolates the visual coefficient where the cue is large. Those rows drag the mean to −0.055 at a truth of +0.05, while the median is right.

**Remedies** (always applied identically to both models of a contrast):

| variant | scoring |
|---|---|
| `sp` | `spline+proj` as validated in notebook 04 (Gaussian log score) |
| `sp+robust` | predictive = 0.99 N(μ, σ²) + 0.01 Cauchy(μ, 4σ), still a proper predictive, so the estimand stays the mean log-score contrast *of that predictive*; per-row loss grows only logarithmically with the error |
| `sp+support` | rows are scored only if hour and T lie inside the 0.5–99.5 % quantile box of that fold's training rows; the estimand becomes information on the supported context region; the excluded share is reported |
| `sp+robust+support` | both |

In every world used here, per-row information does not depend on the context, so restricting to the support leaves the truth unchanged. In real data it may not, which is why the excluded share is part of the result.

**Protocol.** Development worlds are new and built to provoke tails (T anomaly SD 4.5 K, 80 days, seeds 7000+). Criteria for a fresh validation suite (seeds ≥ 120 000, worlds not used in notebooks 03–05 development) are declared in §3 before it runs.

In [1]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from visual_hygrometry import inference, policy, scoring, splits, synthetic, worlds
from visual_hygrometry.worlds import HOUR, frame, gauss_world, regime

pd.set_option('display.precision', 4)
par = lambda: Parallel(n_jobs=16, max_nbytes=None)
KNOTS = np.arange(-4, 45, 4.0)  # °C, as in notebook 04
VARIANTS = {'sp': (False, False), 'sp+robust': (True, False), 'sp+support': (False, True), 'sp+robust+support': (True, True)}


def nonlin(dterm, vterm, v_sd, gamma=0.0):
    def f(df, rng):
        T, h = df.temp_cond_c.to_numpy(), df.hour.to_numpy()
        df['d_ref_k'] = df.d_ref_k + dterm(T, h)
        df['vis_ctx'] = vterm(T, h) + gamma * df.dres + rng.normal(0, v_sd, len(df))
        return df
    return f


def one_rep(W, seed, variants=tuple(VARIANTS), n_pop=200, n_perm=39):
    n = W['n_days']
    df = frame(W, n + n_pop, seed)
    g, slot = df.day.to_numpy(), df.groupby('day').cumcount().to_numpy()
    s = g < n
    raw = df[['hour_local', 'temp_cond_c']].to_numpy()
    df, ctx = worlds.add_context_basis(df, KNOTS)
    Vhat = inference.project(raw, df[W['vis']].to_numpy(), g)
    for j, v in enumerate(W['vis']): df[f'ctxproj_{v}'] = Vhat[:, j]
    Xc, Xvc, y, _ = policy.select_sci(df, 'e2', ctx + [f'ctxproj_{v}' for v in W['vis']], W['vis'], allow_shared_sensor=W.get('shared', False))
    C, X, y = Xc.to_numpy(), Xvc.to_numpy(), y.to_numpy()
    sp = splits.kfold(splits.assign_folds(g[s], 5))
    half = splits.assign_folds(g[s], 2)
    fit = np.zeros(len(g), bool)
    fit[np.flatnonzero(s)[half == 0]] = True
    keep_cf, keep_fz = scoring.support_mask(raw[s], sp), scoring.support_mask(raw, [(fit, ~fit)])
    out = {'seed': seed, 'excluded': 1 - keep_cf.mean()}
    for name in variants:
        robust, support = VARIANTS[name]
        kc, kz = (keep_cf, keep_fz) if support else (None, None)
        r = scoring.contrast(C[s], X[s], y[s], sp, g[s], keep=kc, robust=robust)
        _, p, _ = inference.perm_test(C[s], X[s, C.shape[1]:], y[s], sp, g[s], slot[s], n_perm, seed + 500_000, keep=kc, robust=robust)
        fz = scoring.contrast(C, X, y, [(fit, ~fit)], g, keep=kz, robust=robust)
        dz = np.full(len(g), np.nan)
        dz[fz.scored] = fz.d
        ev = s & ~fit & np.isfinite(dz)
        lo, hi = inference.cluster_interval(dz[ev], g[ev])
        out |= {f'{name}:est': r.mean, f'{name}:p': p, f'{name}:frz_lo1': inference.cluster_lower(dz[ev], g[ev]), f'{name}:frz_lo': lo, f'{name}:frz_hi': hi,
                f'{name}:frz_target': np.nanmean(dz[~s]), f'{name}:min_row': r.d.min()}
    return out


def run(W, seeds, variants=tuple(VARIANTS)): return pd.DataFrame(par()(delayed(one_rep)(W, s, variants) for s in seeds))


def summarise(name, W, a, variants=tuple(VARIANTS)):
    t, rows = W['truth'], []
    for v in variants:
        e = a[f'{v}:est']
        rows.append(dict(world=name, variant=v, truth=t, bias=e.mean() - t, median_bias=e.median() - t, sd_est=e.std(), worst_row=a[f'{v}:min_row'].min(),
                         rej_perm=(a[f'{v}:p'] <= .05).mean(), rej_frz=(a[f'{v}:frz_lo1'] > 0).mean(),
                         cov_frz=((a[f'{v}:frz_lo'] <= t) & (a[f'{v}:frz_hi'] >= t)).mean(),
                         cov_frz_target=((a[f'{v}:frz_lo'] <= a[f'{v}:frz_target']) & (a[f'{v}:frz_hi'] >= a[f'{v}:frz_target'])).mean(),
                         excluded=a.excluded.mean() if 'support' in v else 0.0))
    return rows

## 1 · Development worlds

80 days, 48 daylight samples per day, T anomaly SD 4.5 K (≈ 2–38 °C), 100 replicates each.

| world | structure | truth |
|---|---|---|
| N1w | well-specified null | 0 |
| N4w | multimodal (regime 2.5 / 0.7 K) null | 0 |
| N3w | `D += 0.15 (T−20)²`, cue `(T−20)²/10 + noise(0.5)` | 0 |
| N8 | `D += 0.6 exp((T−20)/5)`, cue `exp((T−20)/5) + noise(0.3)` | 0 |
| P1w | plain cue | 0.05 |
| P3w | quadratic in D, cue = quadratic + moisture signal | 0.05 |
| P5 | exponential in D, cue = exponential + moisture signal | 0.10 |

In [2]:
dw = lambda gamma=0.0, **kw: gauss_world(gamma, **({'t_sd': 4.5} | kw))
mixed_gamma = lambda c, v_sd, r_sd=2.0: np.sqrt(np.expm1(2 * c)) * v_sd / r_sd
quad = lambda T, h: 0.15 * (T - 20) ** 2
expo = lambda T, h: 0.6 * np.exp((T - 20) / 5)
DEV = {'N1w': dict(spec=dw(), vis=['vis_moist'], truth=0.0),
       'N4w': dict(spec=dw(), vis=['vis_moist'], truth=0.0, modify=regime((2.5, 0.7))),
       'N3w': dict(spec=dw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(quad, lambda T, h: (T - 20) ** 2 / 10, .5)),
       'N8': dict(spec=dw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(expo, lambda T, h: expo(T, h) / 0.6, .3)),
       'P1w': dict(spec=dw(.162), vis=['vis_moist'], truth=synthetic.cmi(dw(.162), 'd_ref_k', 'vis_moist', 'temp_cond_c')),
       'P3w': dict(spec=dw(), vis=['vis_ctx'], truth=0.05, modify=nonlin(quad, lambda T, h: (T - 20) ** 2 / 10, .5, mixed_gamma(.05, .5))),
       'P5': dict(spec=dw(), vis=['vis_ctx'], truth=0.10, modify=nonlin(expo, lambda T, h: expo(T, h) / 0.6, .3, mixed_gamma(.10, .3)))}
DEV = {k: W | {'n_days': 80} for k, W in DEV.items()}
dev_raw = {k: run(W, range(7000, 7100)) for k, W in DEV.items()}
dev = pd.DataFrame([r for k, a in dev_raw.items() for r in summarise(k, DEV[k], a)]).set_index(['world', 'variant'])
dev

truth    bias  median_bias  sd_est  worst_row  \
world variant                                                             
N1w   sp                 0.0000  0.0002  -1.1000e-03  0.0176    -2.1473   
      sp+robust          0.0000 -0.0015  -1.0849e-03  0.0028    -2.1219   
      sp+support         0.0000 -0.0015  -1.0689e-03  0.0028    -1.2801   
      sp+robust+support  0.0000 -0.0015  -1.1067e-03  0.0027    -1.1540   
N4w   sp                 0.0000  0.1132  -1.6560e-03  1.1627   -21.0984   
      sp+robust          0.0000 -0.0012  -1.5707e-03  0.0079    -5.2765   
      sp+support         0.0000 -0.0015  -1.6070e-03  0.0032    -2.5455   
      sp+robust+support  0.0000 -0.0015  -1.5457e-03  0.0028    -1.1736   
N3w   sp                 0.0000  0.0082   2.2320e-04  0.0699  -225.7696   
      sp+robust          0.0000  0.0010   1.3610e-04  0.0059    -9.2273   
      sp+support         0.0000  0.0003  -2.1943e-04  0.0033    -1.0978   
      sp+robust+support  0.0000  0.0003  -1.9728e-04  0.0034    -1.0624   
N8    sp                 0.0000  0.0214   1.7300e-03  0.0805   -17.3256   
      sp+robust          0.0000  0.0040   1.2973e-03  0.0150    -4.9054   
      sp+support         0.0000  0.0018  -2.8899e-05  0.0129    -1.8436   
      sp+robust+support  0.0000  0.0018   2.7855e-05  0.0129    -1.7889   
P1w   sp                 0.0499  0.0664   1.0163e-03  0.6491   -15.6528   
      sp+robust          0.0499  0.0024   1.6937e-03  0.0186    -7.0498   
      sp+support         0.0499  0.0018   1.4850e-03  0.0176    -4.6983   
      sp+robust+support  0.0499  0.0019   1.5931e-03  0.0177    -3.4252   
P3w   sp                 0.0500  0.0489   1.1229e-02  0.1614  -224.8780   
      sp+robust          0.0500  0.0125   8.9771e-03  0.0244    -7.0539   
      sp+support         0.0500  0.0063   3.1775e-03  0.0215    -3.4964   
      sp+robust+support  0.0500  0.0064   3.3213e-03  0.0215    -2.8263   
P5    sp                 0.1000 -0.0944  -4.9415e-04  0.6539 -2274.2155   
      sp+robust          0.1000 -0.0023  -3.5752e-04  0.0197    -9.3450   
      sp+support         0.1000 -0.0014  -6.7119e-04  0.0164    -4.8028   
      sp+robust+support  0.1000 -0.0016  -9.3628e-04  0.0162    -3.6609   

                         rej_perm  rej_frz  cov_frz  cov_frz_target  excluded  
world variant                                                                  
N1w   sp                     0.00     0.05     0.87            0.91    0.0000  
      sp+robust              0.00     0.05     0.87            0.92    0.0000  
      sp+support             0.01     0.05     0.87            0.95    0.0149  
      sp+robust+support      0.01     0.03     0.87            0.95    0.0149  
N4w   sp                     0.04     0.02     0.90            0.89    0.0000  
      sp+robust              0.04     0.00     0.89            0.94    0.0000  
      sp+support             0.04     0.02     0.92            0.93    0.0149  
      sp+robust+support      0.04     0.00     0.91            0.94    0.0149  
N3w   sp                     0.04     0.09     0.87            0.54    0.0000  
      sp+robust              0.03     0.09     0.88            0.78    0.0000  
      sp+support             0.06     0.07     0.84            0.90    0.0149  
      sp+robust+support      0.06     0.07     0.84            0.90    0.0149  
N8    sp                     0.06     0.12     0.84            0.49    0.0000  
      sp+robust              0.06     0.14     0.83            0.65    0.0000  
      sp+support             0.10     0.11     0.80            0.90    0.0149  
      sp+robust+support      0.09     0.12     0.78            0.89    0.0149  
P1w   sp                     1.00     0.75     0.95            0.89    0.0000  
      sp+robust              1.00     0.77     0.95            0.92    0.0000  
      sp+support             1.00     0.74     0.94            0.91    0.0149  
      sp+robust+support      1.00     0.75     0.94            0.92    0.0149  
P3w   sp       

## 2 · Reading the development suite

*Written after the development run, before the validation run.*

- **The tail failure is reproduced:** under `sp` the worst held-out rows score −224 (N3w), −398 (P3w) and −5983 nats (P5). The mean is wrecked (P5 bias −0.18, SD 0.93) while the median is right (−0.003).
- **`sp+robust`** bounds the rows (worst −9.3 nats) and fixes most of the mean bias, but leaves +0.011 in P3w and frozen target coverage 0.68–0.81.
- **`sp+support`** removes the extrapolated rows (1.5 % excluded) and does better: worst row −4.9, mixed-cue bias +0.006, frozen target coverage 0.88–0.94.
- **`sp+robust+support`** is the best on every row of the table: worst row −3.8; |bias| ≤ 0.002 except P3w (+0.006); permutation Type I 0.01–0.03; power 1.00; frozen target coverage 0.88–0.94. **Chosen.**
- **Not fixed by any variant:** the frozen one-sided test rejects the exponential null N8 in 0.12 of replicates. Frozen coverage of the *truth* is 0.70–0.85 in the mixed and exponential worlds. Models fitted on half of the days capture less than the full information, so the frozen interval targets something smaller than the truth. That interval is honest about its target but is not an interval for E2.

## 3 · Fresh validation suite

**Chosen method:** `sp+robust+support`, with `sp` alongside for reference. **Criteria, declared before the first validation run**, at the standard of notebook 04 (150 replicates per world):

| id | criterion |
|---|---|
| C1 | permutation Type I ≤ 0.085 in each null Z-N1 … Z-N6 |
| C2 | frozen Type I ≤ 0.085 in each null |
| C3 | permutation power ≥ 0.80 in Z-P1, Z-P3, Z-P5 |
| C4 | |mean bias| ≤ 0.01 in Z-P1, Z-P2 (plain) and ≤ 0.015 in Z-P3 … Z-P5 (mixed) |
| C5 | frozen interval covers the truth ≥ 0.88 and its own target ≥ 0.88 in every positive world |
| C6 | worst single held-out row ≥ −10 nats over all replicates and worlds |
| C7 | excluded (off-support) share ≤ 0.05 |

Worlds (none used before): 30 samples/day over 07–18.5 h, 100 days, β = 0.45, T anomaly SD 4.2 K, moisture residual SD 1.6 K (55 % day-level, AR 0.88), visual noise 35 % day-level (AR 0.65); seeds 120 000+.

| world | structure | truth |
|---|---|---|
| Z-N1 | well-specified null | 0 |
| Z-N2 | regime 1.9 / 0.55 K (p = 0.45), strong serial dependence | 0 |
| Z-N3 | hinge `0.5 max(T−24, 0)`, encoded | 0 |
| Z-N4 | cubic `0.004 (T−19)³`, encoded | 0 |
| Z-N5 | softplus `2 log(1 + e^{(T−22)/2})`, encoded | 0 |
| Z-N6 | bilinear `0.08 (T−20)(h−12)`, encoded | 0 |
| Z-P1, Z-P2 | plain cue | 0.05, 0.10 |
| Z-P3 / Z-P4 / Z-P5 | mixed: hinge / cubic / softplus in D, cue = term + moisture signal | 0.05 / 0.10 / 0.05 |

In [3]:
ZP = dict(per_day=30, hours=(7, 18.5), n_days=100)
zw = lambda gamma=0.0, **kw: gauss_world(gamma, **({'beta': .45, 't_sd': 4.2, 'r_sd': 1.6, 'r_day': .55, 'r_ar': .88, 'v_day': .35, 'v_ar': .65} | kw))
hinge = lambda T, h: 0.5 * np.maximum(T - 24, 0)
cubic = lambda T, h: 0.004 * (T - 19) ** 3
softp = lambda T, h: 2 * np.logaddexp(0, (T - 22) / 2)
bilin = lambda T, h: 0.08 * (T - 20) * (h - 12)
mg = lambda c, v_sd: mixed_gamma(c, v_sd, 1.6)
g05 = np.sqrt(np.expm1(.1)) / 1.6
g10 = np.sqrt(np.expm1(.2)) / 1.6
VAL = {'Z-N1': dict(spec=zw(), vis=['vis_moist'], truth=0.0),
       'Z-N2': dict(spec=zw(r_day=.8, r_ar=.96), vis=['vis_moist'], truth=0.0, modify=regime((1.9, .55), .45)),
       'Z-N3': dict(spec=zw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(hinge, lambda T, h: hinge(T, h) / .5, .3)),
       'Z-N4': dict(spec=zw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(cubic, lambda T, h: cubic(T, h) / .4, .3)),
       'Z-N5': dict(spec=zw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(softp, lambda T, h: softp(T, h) / 2, .3)),
       'Z-N6': dict(spec=zw(), vis=['vis_ctx'], truth=0.0, modify=nonlin(bilin, lambda T, h: bilin(T, h) / .8, .3)),
       'Z-P1': dict(spec=zw(g05), vis=['vis_moist'], truth=synthetic.cmi(zw(g05), 'd_ref_k', 'vis_moist', 'temp_cond_c')),
       'Z-P2': dict(spec=zw(g10), vis=['vis_moist'], truth=synthetic.cmi(zw(g10), 'd_ref_k', 'vis_moist', 'temp_cond_c')),
       'Z-P3': dict(spec=zw(), vis=['vis_ctx'], truth=0.05, modify=nonlin(hinge, lambda T, h: hinge(T, h) / .5, .3, mg(.05, .3))),
       'Z-P4': dict(spec=zw(), vis=['vis_ctx'], truth=0.10, modify=nonlin(cubic, lambda T, h: cubic(T, h) / .4, .3, mg(.10, .3))),
       'Z-P5': dict(spec=zw(), vis=['vis_ctx'], truth=0.05, modify=nonlin(softp, lambda T, h: softp(T, h) / 2, .3, mg(.05, .3)))}
VAL = {k: ZP | W for k, W in VAL.items()}
CHOSEN = ('sp', 'sp+robust+support')
val_raw = {k: run(W, range(120000, 120150), CHOSEN) for k, W in VAL.items()}
val = pd.DataFrame([r for k, a in val_raw.items() for r in summarise(k, VAL[k], a, CHOSEN)]).set_index(['world', 'variant'])
val

truth        bias  median_bias  sd_est   worst_row  \
world variant                                                                 
Z-N1  sp                  0.00  9.3513e-04      -0.0016  0.0360     -7.2167   
      sp+robust+support   0.00 -2.0649e-03      -0.0019  0.0038     -1.3826   
Z-N2  sp                  0.00  1.3042e-03      -0.0028  0.0355    -23.6721   
      sp+robust+support   0.00 -2.7054e-03      -0.0030  0.0047     -1.7618   
Z-N3  sp                  0.00  8.3635e-04      -0.0001  0.0039    -10.0524   
      sp+robust+support   0.00 -4.5599e-04      -0.0003  0.0011     -1.1053   
Z-N4  sp                  0.00  1.8615e-02       0.0031  0.1161   -694.9724   
      sp+robust+support   0.00  1.3177e-03       0.0001  0.0050     -3.1493   
Z-N5  sp                  0.00 -1.6847e-02      -0.0004  0.1495  -2003.4027   
      sp+robust+support   0.00 -7.9342e-04      -0.0007  0.0013     -1.0266   
Z-N6  sp                  0.00 -2.8850e-04      -0.0003  0.0016    -11.7136   
      sp+robust+support   0.00 -3.0050e-04      -0.0003  0.0008     -0.7938   
Z-P1  sp                  0.05 -3.0551e-03      -0.0043  0.0430   -539.2459   
      sp+robust+support   0.05 -1.0077e-03      -0.0039  0.0199     -3.4423   
Z-P2  sp                  0.10  9.4386e-03      -0.0022  0.1335   -219.1133   
      sp+robust+support   0.10 -3.6741e-05      -0.0031  0.0270     -4.1749   
Z-P3  sp                  0.05 -1.7721e-01      -0.0007  1.3110 -10930.8284   
      sp+robust+support   0.05 -2.8709e-04      -0.0008  0.0084     -2.7859   
Z-P4  sp                  0.10 -2.5319e-01      -0.0080  1.2486  -9300.3957   
      sp+robust+support   0.10 -9.3174e-03      -0.0063  0.0210     -3.7072   
Z-P5  sp                  0.05 -1.3233e-03       0.0036  0.0776   -965.0802   
      sp+robust+support   0.05  1.7523e-03       0.0018  0.0076     -2.5235   

                         rej_perm  rej_frz  cov_frz  cov_frz_target  excluded  
world variant                                                                  
Z-N1  sp                   0.0533   0.0200   0.9267          0.9200    0.0000  
      sp+robust+support    0.0600   0.0267   0.9133          0.9067    0.0149  
Z-N2  sp                   0.0533   0.0200   0.9133          0.8733    0.0000  
      sp+robust+support    0.0467   0.0267   0.9133          0.9333    0.0149  
Z-N3  sp                   0.0600   0.0533   0.9200          0.7667    0.0000  
      sp+robust+support    0.0533   0.0467   0.9067          0.9200    0.0149  
Z-N4  sp                   0.0533   0.2133   0.8800          0.6067    0.0000  
      sp+robust+support    0.0533   0.1667   0.8333          0.9133    0.0149  
Z-N5  sp                   0.0400   0.0267   0.9133          0.7600    0.0000  
      sp+robust+support    0.0267   0.0200   0.8467          0.9267    0.0149  
Z-N6  sp                   0.0200   0.0933   0.8867          0.8333    0.0000  
      sp+robust+support    0.0333   0.0800   0.9000          0.8933    0.0149  
Z-P1  sp                   0.9867   0.7000   0.9067          0.9067    0.0000  
      sp+robust+support    1.0000   0.7400   0.8933          0.9267    0.0149  
Z-P2  sp                   0.9933   0.9000   0.9200          0.9133    0.0000  
      sp+robust+support    1.0000   0.9267   0.9067          0.9267    0.0149  
Z-P3  sp                   0.8867   0.8067   0.8533          0.7667    0.0000  
      sp+robust+support    1.0000   0.9867   0.8200          0.9200    0.0149  
Z-P4  sp                   0.7333   0.6467   0.8267          0.5867    0.0000  
      sp+robust+support    1.0000   0.9800   0.7467          0.9133    0.0149  
Z-P5  sp                   0.9533   0.9000   0.9067          0.7733    0.0000  
      sp+robust+support    1.0000   1.0000   0.8933          0.9267    0.0149

In [4]:
v = lambda w, c, var='sp+robust+support': val.loc[(w, var), c]
nulls, plain, mixed = [f'Z-N{i}' for i in range(1, 7)], ['Z-P1', 'Z-P2'], ['Z-P3', 'Z-P4', 'Z-P5']
acc = [('C1', 'perm Type I <= 0.085, nulls', [v(w, 'rej_perm') for w in nulls], all(v(w, 'rej_perm') <= .085 for w in nulls)),
       ('C2', 'frozen Type I <= 0.085, nulls', [v(w, 'rej_frz') for w in nulls], all(v(w, 'rej_frz') <= .085 for w in nulls)),
       ('C3', 'perm power >= 0.80, Z-P1/P3/P5', [v(w, 'rej_perm') for w in ('Z-P1', 'Z-P3', 'Z-P5')], all(v(w, 'rej_perm') >= .8 for w in ('Z-P1', 'Z-P3', 'Z-P5'))),
       ('C4', '|bias| <= 0.01 plain, <= 0.015 mixed', [v(w, 'bias') for w in plain + mixed], all(abs(v(w, 'bias')) <= .01 for w in plain) and all(abs(v(w, 'bias')) <= .015 for w in mixed)),
       ('C5', 'frozen covers truth and target >= 0.88', [(v(w, 'cov_frz'), v(w, 'cov_frz_target')) for w in plain + mixed], all(v(w, 'cov_frz') >= .88 and v(w, 'cov_frz_target') >= .88 for w in plain + mixed)),
       ('C6', 'worst row >= -10 nats', min(v(w, 'worst_row') for w in VAL), min(v(w, 'worst_row') for w in VAL) >= -10),
       ('C7', 'excluded share <= 0.05', max(v(w, 'excluded') for w in VAL), max(v(w, 'excluded') for w in VAL) <= .05)]
pd.DataFrame([(i, c, np.round(x, 3), p) for i, c, x, p in acc], columns=['id', 'criterion', 'value', 'pass'])

,id,criterion,value,pass
0,C1,"perm Type I <= 0.085, nulls","[0.06, 0.047, 0.053, 0.053, 0.027, 0.033]",True
1,C2,"frozen Type I <= 0.085, nulls","[0.027, 0.027, 0.047, 0.167, 0.02, 0.08]",False
2,C3,"perm power >= 0.80, Z-P1/P3/P5","[1.0, 1.0, 1.0]",True
3,C4,"|bias| <= 0.01 plain, <= 0.015 mixed","[-0.001, -0.0, -0.0, -0.009, 0.002]",True
4,C5,frozen covers truth and target >= 0.88,"[[0.893, 0.927], [0.907, 0.927], [0.82, 0.92],...",False
5,C6,worst row >= -10 nats,-4.175,True
6,C7,excluded share <= 0.05,0.015,True


**Validation outcome: C1, C3, C4, C6, C7 pass; C2 and C5 fail.**

- **Fixed: the tails.** The worst single row over all 1650 validation replicates is −4.2 nats (`sp`: −10 931). Mean bias is ≤ 0.001 on plain cues and ≤ 0.009 on mixed cues, where `sp` alone gave −0.18 and −0.25. Only 1.5 % of rows fall off-support.
- **Still valid: the zero-information test.** Permutation Type I is 0.027–0.060 in six fresh nulls (hinge, cubic, softplus, bilinear, regime with strong serial dependence), with power 1.00.
- **Not solved: an interval for E2.** The frozen hold-out interval covers *its own target* in 0.91–0.93 of replicates, but that target is the information captured by models fitted on half of the days. It is below E2 in the mixed worlds (truth coverage 0.75–0.82, C5). Under the cubic null it is even positive, so the frozen one-sided test rejects 0.167 (C2). The frozen test is therefore not usable for Q0, which already rests on the permutation test, and the frozen interval is not an interval for E2.

The test (Q0) and the point estimate now pass on worlds never used in development. The remaining question is which interval to report for the effect size, and against which estimand (E2 on the supported region vs. information captured by a fitted model). That is a methodological decision, not a bug, and it goes to independent review (`reviews/m0_revision2_review.md`).